# The LangGraph Supervisor Agent

Three stores, one question type each. Genie answers from the sensor readings in
Delta, Cypher answers from the fleet graph, GraphRAG answers from the maintenance
manuals. Here you build the supervisor that picks between them.

**Prerequisites**

| Lab | What this notebook needs from it |
|---|---|
| [Lab 2](../Lab_2_Databricks_ETL_Neo4j) | The fleet graph in your own Aura instance |
| [Lab 1 notebook 02](../Lab_1_Aura_Setup/02_credentials_and_cypher.ipynb) | The `fleet-ops-<your-user>` secret scope holding your Aura credentials |
| [Lab 3 notebook 01](../Lab_3_Semantic_Search/01_data_and_embeddings.ipynb) | The `maintenanceChunkEmbeddings` vector index |
| [Lab 3 notebook 02](../Lab_3_Semantic_Search/02_graphrag_retrievers.ipynb) | The `VectorCypherRetriever` this notebook's manual tool is built from |
| [Lab 4 Part A](../Lab_4_Compound_AI_Agents/04_genie_agent.ipynb) | Your Genie space, and its space ID |

**Learning objectives**

- Build three tools over three stores
- Write a supervisor prompt that separates the two that look alike
- Wire them into a LangGraph `StateGraph` with a routing loop
- Run one question that needs all three, and read the route it took
- Measure routing accuracy, and measure the hard pair on its own

## The shape of the agent

```
                    question
                       |
                       v
              +------------------+
              |    supervisor    |<---------+
              | Claude Sonnet 5  |          |
              +------------------+          |
                       |                    |
        +--------------+--------------+     |
        v              v              v     |
  +-----------+  +-----------+  +----------------+
  | genie     |  | cypher    |  | graphrag       |
  | Delta     |  | Neo4j     |  | Neo4j vector   |
  | telemetry |  | traversal |  | + Cypher tail  |
  +-----------+  +-----------+  +----------------+
        |              |              |     |
        +--------------+--------------+-----+
                       |
                       v
                  synthesize --> answer
```

Each tool reports back to the supervisor, which calls another tool or stops. That
loop is what lets one question use three tools in sequence, with each result
informing the next choice.

The prompt is the hard part, not the wiring. `cypher_node` and `graphrag_node`
both end in a graph traversal, so a model that describes tools by their ending
cannot tell them apart. Section 6 is about that.

## Section 1: Configuration

One thing to set: your Genie space ID from Lab 4 Part A.

Everything else, the Neo4j database name included, comes from the secret scope
Lab 3 notebook 01 created. The scope name derives from `current_user()`, so it is
the same in both notebooks and nothing needs copying across.

In [ ]:
# ==================================================
# CONFIGURATION - replace GENIE_AGENT_ID with yours
# ==================================================

# From Lab 4 Part A. Open your Genie space and take the ID out of the URL:
#   https://<workspace>/genie/rooms/<GENIE_AGENT_ID>
GENIE_AGENT_ID = "01f1661b55731a0293c3f84ac9c5ba52"

import sys

sys.path.insert(0, ".")

from tools import secret_scope_name

SECRET_SCOPE = secret_scope_name(spark)
print(f"Secret scope:   {SECRET_SCOPE}")
print(f"Genie space ID: {GENIE_AGENT_ID}")

## Section 2: Connections

`tools.py` sits next to this notebook and holds the node builders, the prompts,
and the graph schema. It imports the embedder and the LLM from Lab 3's
`data_utils.py`, because a query has to be embedded by the same model that wrote
the vectors in `maintenanceChunkEmbeddings` to match them.

In [ ]:
from databricks.sdk import WorkspaceClient

from tools import (
    EMBEDDING_ENDPOINT,
    LLM_ENDPOINT,
    VECTOR_INDEX_NAME,
    build_cypher_node,
    build_genie_node,
    build_graphrag_node,
    build_supervisor_node,
    build_synthesize_node,
    get_embedder,
    get_llm,
    open_driver_from_secrets,
    read_neo4j_secrets,
    vector_index_exists,
)

# The password is used and dropped inside the helper, so it never lands in a
# notebook variable.
driver = open_driver_from_secrets(dbutils, SECRET_SCOPE)

# The database name lives in the same scope. A scope written before that key
# existed reads back as "neo4j", the AuraDB Free default.
NEO4J_DATABASE = read_neo4j_secrets(dbutils, SECRET_SCOPE)["database"]

llm = get_llm(LLM_ENDPOINT)
embedder = get_embedder(EMBEDDING_ENDPOINT)
workspace = WorkspaceClient()

# The database name came out of a secret, so printing it prints [REDACTED].
print("Neo4j:      connected")
print(f"Supervisor: {LLM_ENDPOINT}")
print(f"Embedder:   {EMBEDDING_ENDPOINT}")

In [ ]:
# Counts from your own Aura instance.
records, _, _ = driver.execute_query(
    """
    CALL () { MATCH (n) RETURN count(n) AS nodes }
    CALL () { MATCH ()-[r]->() RETURN count(r) AS rels }
    CALL () { MATCH (c:Chunk) RETURN count(c) AS chunks }
    RETURN nodes, rels, chunks
    """,
    database_=NEO4J_DATABASE,
)
stats = records[0]
print(f"Nodes:         {stats['nodes']:,}")
print(f"Relationships: {stats['rels']:,}")
print(f"Manual chunks: {stats['chunks']:,}")
print(
    f"Vector index '{VECTOR_INDEX_NAME}': "
    f"{'online' if vector_index_exists(driver, VECTOR_INDEX_NAME, NEO4J_DATABASE) else 'MISSING'}"
)

## Section 3: `genie_node`, the telemetry tool

Your Genie space is already an agent. English question in, SQL against the four
Lakehouse tables, results explained. Wrapping it as a tool means starting a
conversation and reading back both parts of the reply: the prose answer, and the
attachment holding the generated SQL and its rows. Both go into the finding, so
the supervisor can pass exact numbers to the next tool.

This is the only tool that can see a sensor reading. Your graph has `Sensor`
nodes but no readings on them. The 155,000 timestamped values live in Delta,
where scanning them is cheap.

In [ ]:
genie_node = build_genie_node(GENIE_AGENT_ID, workspace)

# One call, before any routing.
probe = genie_node({"question": "What is the average EGT for aircraft N10000?"})
print(probe["findings"][-1]["content"][:1200])

## Section 4: `cypher_node`, the graph tool

Text to Cypher, against your own Aura instance over Bolt. The LLM gets the schema
and the question, writes a query, and the query runs.

- **The schema was measured, not remembered.** `tools.GRAPH_SCHEMA` lists what
  Lab 2 loaded plus what Lab 3 added, with the exact labels and property spellings
  in your graph. A schema that promises something the graph lacks produces Cypher
  that runs cleanly, returns zero rows, and leaves the agent reporting it found
  nothing.
- **Writes are blocked twice.** A regular expression check first, so the refusal
  is a sentence rather than a driver error, then a read transaction, where Aura
  rejects the write itself.
- **A failed query gets one retry with its error attached.** Most text to Cypher
  failures are a mistyped property or a relationship pointing the wrong way, and
  the error message says which.

In [ ]:
cypher_node = build_cypher_node(driver, llm, database=NEO4J_DATABASE)

probe = cypher_node(
    {"question": "Which aircraft have had critical maintenance events on their engine systems, and which components were involved?"}
)
print(probe["findings"][-1]["content"][:1500])

One trap to know about. `OperatingLimit` holds the ceilings a manual sets for an
aircraft model, not anything a sensor measured, so the schema tells this tool to
refuse reading questions outright rather than return the nearest number it can
find. Those questions belong to Genie.

## Section 5: `graphrag_node`, the manual tool

The Lab 3 notebook 02 retriever, wrapped as a node. The question is embedded with
`databricks-bge-large-en`, the vector index returns the closest manual chunks, and
a Cypher tail runs from each hit.

The tail is what makes this more than vector search:

```cypher
WITH node
OPTIONAL MATCH (previous:Chunk)-[:NEXT_CHUNK]->(node)
OPTIONAL MATCH (node)-[:NEXT_CHUNK]->(following:Chunk)
MATCH (node)-[:FROM_DOCUMENT]->(doc:Document)
OPTIONAL MATCH (doc)-[:APPLIES_TO]->(a:Aircraft)-[:HAS_SYSTEM]->(s:System)
```

Sideways along `NEXT_CHUNK`, so a procedure split across a chunk boundary arrives
in one piece. Up through the `Document` to the aircraft and its systems, so the
answer knows which tail numbers it covers. Neither is in the embedding. Both are
one hop away in the graph.

If the vector index is missing, the builder returns a node that says so instead
of raising. The agent still runs, with two tools rather than three.

In [ ]:
graphrag_node = build_graphrag_node(
    driver, llm, embedder, database=NEO4J_DATABASE, top_k=3
)

if getattr(graphrag_node, "available", False):
    probe = graphrag_node(
        {"question": "What does the manual say about EGT exceedance during takeoff?"}
    )
    print(probe["findings"][-1]["content"][:1500])
else:
    print("Manual tool unavailable. This is what the supervisor would receive:\n")
    print(graphrag_node({"question": "anything"})["findings"][-1]["content"])

## Section 6: The supervisor prompt

`cypher_node` and `graphrag_node` both end in a Neo4j traversal. Describe them by
what they do at the end and the model sends manual questions to Cypher, where
they return nothing, because manual text is not a property it can filter on.

So the prompt decides on where the question **starts**:

> Starts with a name you could put in a `WHERE` clause -> `cypher_node`
> Starts with a phrase you would search a manual for -> `graphrag_node`

| Question | Route | Why |
|---|---|---|
| "What maintenance events did N10004 have?" | `cypher_node` | N10004 is a node |
| "What is the procedure for an EGT exceedance?" | `graphrag_node` | That is a phrase in a manual, not a node |
| "What is the documented EGT limit for the A320-200?" | `cypher_node` | A `maxValue` property on an `OperatingLimit` |
| "How do I troubleshoot engine vibration?" | `graphrag_node` | A procedure, so it lives in the manual text |

Four rules about stopping go with it. A supervisor that loops back to itself will
call the same tool three times unless told not to, because calling a tool again
always looks safer than answering. The rules cap each tool at one call and say to
synthesize as soon as every part of the question has something against it.
`MAX_TOOL_CALLS` is the backstop under those rules, not a substitute for them.

Read the prompt printed below. It is the part of this lab you are most likely to
change for your own domain.

In [ ]:
from tools import SUPERVISOR_PROMPT

# Without the index, drop graphrag_node so the supervisor stops offering an
# answer it cannot produce.
AVAILABLE_TOOLS = ["genie_node", "cypher_node"]
if getattr(graphrag_node, "available", False):
    AVAILABLE_TOOLS.append("graphrag_node")

supervisor_node = build_supervisor_node(llm, available_tools=AVAILABLE_TOOLS)
synthesize_node = build_synthesize_node(llm)

print(f"Tools the supervisor can call: {', '.join(AVAILABLE_TOOLS)}\n")
print(SUPERVISOR_PROMPT.split("## The question")[0])

## Section 7: Wiring the graph

Five nodes and one decision. `START` goes to the supervisor, the supervisor's
`route` picks a tool or picks `synthesize`, every tool goes back to the
supervisor, `synthesize` goes to `END`.

The edge back from each tool is what makes this a supervisor rather than a
router. A router picks one tool and answers. This one sees what the tool returned
and picks again.

State is a `TypedDict` with five keys. `trace` is the list of tools called in
order, which is the record Section 9 measures.

In [ ]:
from langgraph.graph import END, START, StateGraph

from tools import MAX_TOOL_CALLS, AgentState, route_from_supervisor

builder = StateGraph(AgentState)

builder.add_node("supervisor", supervisor_node)
builder.add_node("genie_node", genie_node)
builder.add_node("cypher_node", cypher_node)
builder.add_node("graphrag_node", graphrag_node)
builder.add_node("synthesize", synthesize_node)

builder.add_edge(START, "supervisor")

# The one decision: route_from_supervisor reads state["route"] and names the
# next node.
builder.add_conditional_edges(
    "supervisor",
    route_from_supervisor,
    {
        "genie_node": "genie_node",
        "cypher_node": "cypher_node",
        "graphrag_node": "graphrag_node",
        "synthesize": "synthesize",
    },
)

# Every tool reports back rather than answering.
for tool_name in ("genie_node", "cypher_node", "graphrag_node"):
    builder.add_edge(tool_name, "supervisor")

builder.add_edge("synthesize", END)

agent = builder.compile()
print(f"Agent compiled. Tool call budget per question: {MAX_TOOL_CALLS}")

In [ ]:
from typing import Any


def ask(question: str, *, verbose: bool = True) -> dict[str, Any]:
    """Run one question through the agent and show the route it took."""
    result = agent.invoke({"question": question, "trace": [], "findings": []})
    if verbose:
        print(f"Q: {question}")
        print(f"Route: {' -> '.join(result['trace']) or '(no tool called)'}\n")
        print(result["answer"])
        print("-" * 78)
    return result

## Section 8: The four routing cases

Three questions that should each land on exactly one tool, then one that needs
all three.

In [ ]:
case_1 = ask("What is the average EGT for aircraft N10000 in August 2024?")

In [ ]:
case_2 = ask("Which aircraft have had critical maintenance events on their engine systems, and which components were involved?")

In [ ]:
case_3 = ask("What does the maintenance manual say about EGT exceedance procedures?")

### The anchor question

One question, three stores. The readings that say which engine runs hot are in
Delta, the maintenance history for that engine is in the graph, the procedure is
in a manual chunk. No single tool answers it, and the supervisor has to work that
out from the question rather than from a rule you wrote.

Watch the route. Three tools in sequence is the point of the lab.

In [ ]:
ANCHOR_QUESTION = (
    "Which engines are showing abnormal EGT readings, what maintenance history "
    "do those aircraft have, and what does the maintenance manual say to do "
    "about high EGT?"
)

anchor = ask(ANCHOR_QUESTION)

## Section 9: Measuring the routing

Three good answers prove the tools work. Three questions is too few to prove the
routing works.

The set below is twelve questions, four per tool, run through the supervisor
alone with no findings to reason from. That is the routing decision in isolation:
what the model picks from the question text on the first call.

`cypher_node` against `graphrag_node` is scored on its own as well. That pair is
where misrouting shows up first, and an overall score hides it.

In [ ]:
ROUTING_CASES = [
    # genie_node: a measured value, or an aggregate over measured values
    ("What is the average EGT for aircraft N10000 in August 2024?", "genie_node"),
    ("Compare average vibration readings between B737-800 and A320-200 aircraft.", "genie_node"),
    ("What was the maximum fuel flow recorded in August 2024?", "genie_node"),
    ("Show the daily trend of N1 speed for aircraft N10000.", "genie_node"),
    # cypher_node: starts from a named entity, answered by following relationships
    ("Which aircraft have had critical maintenance events on their engine systems, and which components were involved?", "cypher_node"),
    ("What maintenance events has aircraft N10004 had, and how severe were they?", "cypher_node"),
    ("Which components are in the hydraulic system of aircraft N10000?", "cypher_node"),
    ("Which aircraft have had the most part removals, and for what reason?", "cypher_node"),
    # graphrag_node: starts from language in a manual
    ("What does the maintenance manual say about EGT exceedance procedures?", "graphrag_node"),
    ("How do I troubleshoot excessive engine vibration?", "graphrag_node"),
    ("What is the documented inspection procedure after a hard landing?", "graphrag_node"),
    ("What steps does the manual give for a hydraulic pressure loss?", "graphrag_node"),
]

results = []
skipped = []
for question, expected in ROUTING_CASES:
    if expected not in AVAILABLE_TOOLS:
        # Scoring a tool the supervisor was never offered measures nothing.
        skipped.append((question, expected))
        continue
    chosen = supervisor_node({"question": question, "trace": [], "findings": []})["route"]
    results.append((question, expected, chosen))
    mark = "PASS" if chosen == expected else "FAIL"
    print(f"{mark}  expected {expected:<14} got {chosen:<14} {question[:52]}")

for question, expected in skipped:
    print(f"SKIP  {expected} is unavailable in this workspace: {question[:52]}")

In [ ]:
def accuracy(rows) -> str:
    if not rows:
        return "n/a"
    hits = sum(1 for _, expected, chosen in rows if expected == chosen)
    return f"{hits}/{len(rows)} ({100 * hits / len(rows):.0f}%)"


graph_pair = [row for row in results if row[1] in ("cypher_node", "graphrag_node")]

print(f"Overall routing accuracy:              {accuracy(results)}")
print()
print(f"  genie_node questions:                {accuracy([r for r in results if r[1] == 'genie_node'])}")
print(f"  cypher_node questions:               {accuracy([r for r in results if r[1] == 'cypher_node'])}")
print(f"  graphrag_node questions:             {accuracy([r for r in results if r[1] == 'graphrag_node'])}")
print()
print(f"cypher_node vs graphrag_node, on its own: {accuracy(graph_pair)}")
print("  This is the number that matters. Both tools end in a traversal, so this")
print("  pair is where a weak routing prompt fails first.")
if skipped:
    print(f"\n{len(skipped)} question(s) skipped because their tool is unavailable here.")

Anything that failed is a prompt problem. Take the question that went wrong, find
the sentence in `SUPERVISOR_PROMPT` that should have caught it, add the pair to
the examples, and rerun the cell. Routing prompts get tuned by looking at what
they got wrong, the same way you would tune a classifier.

If the pair number stays low after a few rounds, try a different model: pass
another endpoint to `get_llm` in the setup cell above. To change it for the whole
course, edit `LLM_ENDPOINT` in `Lab_3_Semantic_Search/data_utils.py` and
`lab/workshop.py`, the two places it is written.

## Section 10: Optional, hybrid retrieval

Skip this unless you ran [Lab 3 notebook
03](../Lab_3_Semantic_Search/03_hybrid_retrievers.ipynb).

`graphrag_node` matches on meaning, which is what you want when the question says
"high exhaust temperature" and the manual says "EGT exceedance". It is the wrong
instrument for an exact string. A part number or engine designation either appears
in the text or it does not, and an embedding of `CFM56-7B` sits close to every
other engine model.

`HybridCypherRetriever` runs the vector index and the `maintenanceChunkText`
fulltext index together and merges the rankings. Same Cypher tail, so the graph
context is unchanged. Add it as a second node, or swap it in for the retriever
inside `build_graphrag_node`.

In [ ]:
from tools import FULLTEXT_INDEX_NAME, MANUAL_CONTEXT_QUERY, format_manual_chunk

records, _, _ = driver.execute_query(
    "SHOW INDEXES YIELD name, type, state "
    "WHERE name = $name AND type = 'FULLTEXT' AND state = 'ONLINE' "
    "RETURN count(*) AS found",
    name=FULLTEXT_INDEX_NAME,
    database_=NEO4J_DATABASE,
)

if records[0]["found"] and getattr(graphrag_node, "available", False):
    from neo4j_graphrag.generation import GraphRAG
    from neo4j_graphrag.retrievers import HybridCypherRetriever

    hybrid_retriever = HybridCypherRetriever(
        driver=driver,
        neo4j_database=NEO4J_DATABASE,
        vector_index_name=VECTOR_INDEX_NAME,
        fulltext_index_name=FULLTEXT_INDEX_NAME,
        retrieval_query=MANUAL_CONTEXT_QUERY,
        embedder=embedder,
        result_formatter=format_manual_chunk,
    )
    hybrid_rag = GraphRAG(llm=llm, retriever=hybrid_retriever)

    answer = hybrid_rag.search(
        "What are the maintenance requirements for the CFM56-7B engine?",
        retriever_config={"top_k": 3},
    )
    print(answer.answer)
else:
    print(
        f"Fulltext index '{FULLTEXT_INDEX_NAME}' not found. "
        "Run Lab 3 notebook 03 to build it, then rerun this cell."
    )

## Section 11: Try your own

Ask it something the workshop did not plan for. Read the route before the answer:
a wrong answer on a sensible route is a tool problem, a wrong answer on a strange
route is a prompt problem, and those get fixed in different files.

Questions worth trying:

- "Which aircraft has the highest vibration, and has it had a maintenance event?"
- "What is the documented N1 speed limit for the B737-800, and how do current readings compare?"
- "Which components on the hydraulics systems have failed, and what does the manual say to check?"

In [ ]:
my_question = "Which aircraft has the highest average vibration, and has it had any maintenance events?"

my_result = ask(my_question)

In [ ]:
driver.close()
print("Neo4j connection closed.")

## What you built

A supervisor over three stores, with the routing rule written down rather than
assumed. The three tools are the three labs before this one. The prompt in
`tools.py` is the only new idea.

Notebook 02 logs the same graph as an MLflow model, deploys it to Model Serving,
and evaluates it against a question set with MLflow's LLM judges. Lab 6 gives it
memory, in Neo4j, so it can take a follow-up.